In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.express as px

import coincidence_v4

folder = r"J:\ctgroup\Edward\DATA\VMI\20251112\4-1,5 PO course scan"

data = pd.DataFrame()
for file in os.listdir(folder):
    if not file.endswith('.cv4'):
        continue
    ind = int(file[:-4])
    print(ind)
    try:
        x, y, t, etof, itof = coincidence_v4.load_file(os.path.join(folder, file))
    except ValueError:
        continue
    temp = pd.DataFrame({'x': x, 'y': y, 't': t, 'etof': etof, 'itof': itof, 'index': ind})
    data = pd.concat([data, temp], ignore_index=True)

In [ ]:
counts = data['index'].value_counts().sort_index()
px.line(
        counts,
        title='Total Counts vs File Index',
        width=800, height=800,
        labels={'index': 'Stage Position (nm)', 'value': 'Total Counts'},
        # log_y=True,
).show()

In [ ]:
data_filt = data[(data['etof'] > 480) & (data['etof'] < 520) & (data['t'] > 246) & (data['t'] < 285)]

h2d, xe, ye = np.histogram2d(
        data_filt['index'],
        data_filt['etof'] + np.random.randn(len(data_filt['index'])) * 0.26,
        bins=(data['index'].nunique(), 256),
        range=((data['index'].min(), data['index'].max()), (480, 520))
)

px.imshow(
        np.log(h2d.T),
        aspect='auto',
        origin='lower',
).show()

data = data.sort_values('index')

lines = []
for i in data_filt['index'].unique():
    ind = data_filt['index'] == i
    etofs = data_filt['etof'][ind].to_numpy()
    lines.append(
            np.histogram(etofs, range=(480, 520), bins=256)[0]
    )

i = 30
px.line(
        y=lines[i]
).show()

px.histogram(
        data[(data['t'] > 0) & (data['t'] < 500)],
        x="t",
        log_y=True
).show()

px.density_heatmap(
        data[(data['etof'] > 480) & (data['etof'] < 520) & (data['t'] > 246) & (data['t'] < 285)],
        x='x',
        y='y',
        nbinsx=256,
        nbinsy=256,
)
data = data_filt

In [ ]:
data.to_hdf(
        os.path.join(folder, "combined.h5"), key='df'
)

In [ ]:
roi = (-np.inf, np.inf)

data_roi = data[(data['index'] > roi[0]) & (data['index'] < roi[1])]
counts_roi = data_roi['index'].value_counts().sort_index()
# add missing values

for i in np.arange(min(counts_roi.index), max(counts_roi.index), min(np.diff(counts_roi.index))):
    if len(counts_roi.loc[counts_roi.index == i]) == 0:
        counts_roi[i] = 0
counts_roi = counts_roi.sort_index()

px.line(
        counts_roi,

        title='Total Counts vs File Index',
        width=800, height=800,
        labels={'index': 'Stage Position (nm)', 'value': 'Total Counts'},
        log_y=False,

).show()
counts_roi = counts_roi.reset_index()

fft = np.fft.fft(counts_roi['count'].to_numpy())
freq = np.fft.fftfreq(n=len(counts_roi['index']), d=np.diff(counts_roi['index'])[0])
wvln = 1 / freq * 2

jacobian_corr = 2 / wvln ** 2

tick_freqs = np.linspace(0, 0.005, 6)
tick_wvlns = 2 / tick_freqs
px.line(
        x=freq[(wvln > 0) & (wvln < 5000)],
        y=np.abs(fft)[(wvln > 0) & (wvln < 5000)],
        labels={'x': 'Wavelength (nm)', 'y': 'Amplitude'},
        width=800, height=800,
        title=f'Fourier Transform: {folder.split("\\")[-1]}',
        range_x=(0, 0.0025)
).update_xaxes(
        tickvals=tick_freqs,
        ticktext=[f"{t:.0f}" for t in tick_wvlns],
).show()

In [ ]:


fft_highpass = np.where(
        np.abs(wvln) < 600,
        0,
        fft
)

px.line(
        x=wvln[(wvln > 0) & (wvln < 5000)],
        y=np.abs(fft_highpass)[(wvln > 0) & (wvln < 5000)] * jacobian_corr[(wvln > 0) & (wvln < 5000)],
).show()

px.line(
        y=np.abs(np.fft.ifft(fft_highpass)),
).add_scatter(
        y=counts_roi['count'],
)